# Phase 7: Multimodal Inference Smoke Test

This notebook validates multimodal inference for Gemma4 + LoRA adapter.

It is designed for quick testing:
- load model + processor once
- run one image+prompt request
- print latency and generated output


In [ ]:
# Kaggle often starts with an older Transformers build that does not support gemma4.
# Run this once per fresh session, then RESTART the kernel.
# %pip install -U "transformers>=4.57.0" "tokenizers>=0.21.0" peft bitsandbytes accelerate pillow

import time
import zipfile
from pathlib import Path

import torch
import transformers
from PIL import Image
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    CONFIG_MAPPING,
)

print("transformers:", transformers.__version__)
if "gemma4" not in CONFIG_MAPPING:
    raise RuntimeError(
        "Your current transformers build does not support gemma4. "
        "Run the pip cell above, then restart the notebook kernel and re-run all cells."
    )


In [ ]:
# Update these paths as needed.
BASE_MODEL = "google/gemma-4-E2B"
ADAPTER_ZIP = Path("/kaggle/input/notebooks/lasdwop/gemma4-tab-multimodal/gemma-4-e2b-multimodal-lora-final.zip")
ADAPTER_DIR = Path("/kaggle/working/gemma-4-e2b-multimodal-lora-final")
IMAGE_PATH = Path("/kaggle/input/datasets/lasdwop/multimodal-dataset/test/images/0.png")

PROMPT = "Complete the missing code shown in this IDE screenshot."

# Runtime knobs for faster smoke tests.
MAX_NEW_TOKENS = 24
MAX_TIME = 30.0
MAX_IMAGE_SIZE = 448
DO_SAMPLE = False
TEMPERATURE = 0.2
TOP_P = 0.9
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 4

# Extract LoRA adapter zip once to a writable location.
if not ADAPTER_DIR.exists():
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ADAPTER_ZIP, "r") as zf:
        zf.extractall(ADAPTER_DIR)

assert ADAPTER_DIR.exists(), f"Missing adapter dir: {ADAPTER_DIR}"
assert IMAGE_PATH.exists(), f"Missing image: {IMAGE_PATH}"


In [ ]:
def build_prompt_text(processor, prompt: str) -> str:
    """Create a multimodal prompt with image token(s)."""
    if hasattr(processor, "apply_chat_template"):
        try:
            return processor.apply_chat_template(
                [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image"},
                            {"type": "text", "text": prompt},
                        ],
                    }
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass

    tok = getattr(processor, "tokenizer", None)
    image_token = "<|image|>"
    if tok is not None:
        image_token = (
            getattr(tok, "image_token", None)
            or tok.special_tokens_map.get("image_token")
            or image_token
        )
    return f"{image_token}\n{prompt}"


In [ ]:
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR), trust_remote_code=True)

# 8-bit load works best for constrained VRAM smoke tests.
model_kwargs = {
    "trust_remote_code": True,
    "low_cpu_mem_usage": True,
    "dtype": torch.float16,
    "device_map": "auto",
    "quantization_config": BitsAndBytesConfig(load_in_8bit=True),
}

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=False)

model.eval()
print("Model loaded.")


In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
if MAX_IMAGE_SIZE > 0:
    image = image.copy()
    image.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.BICUBIC)

prompt_text = build_prompt_text(processor, PROMPT)
inputs = processor(
    text=[prompt_text],
    images=[image],
    return_tensors="pt",
    padding=True,
    truncation=True,
)

target_device = getattr(model, "device", torch.device("cpu"))
inputs = {k: v.to(target_device) for k, v in inputs.items()}

gen_kwargs = {
    "max_new_tokens": int(MAX_NEW_TOKENS),
    "do_sample": bool(DO_SAMPLE),
    "repetition_penalty": float(REPETITION_PENALTY),
    "no_repeat_ngram_size": int(NO_REPEAT_NGRAM_SIZE),
    "max_time": float(MAX_TIME),
    "eos_token_id": processor.tokenizer.eos_token_id,
    "pad_token_id": processor.tokenizer.pad_token_id,
}
if DO_SAMPLE:
    gen_kwargs["temperature"] = float(TEMPERATURE)
    gen_kwargs["top_p"] = float(TOP_P)

t0 = time.time()
with torch.no_grad():
    out = model.generate(**inputs, **gen_kwargs)
elapsed_ms = int((time.time() - t0) * 1000)

prompt_len = inputs["input_ids"].shape[-1]
new_tokens = out[0][prompt_len:]
text = processor.tokenizer.decode(new_tokens, skip_special_tokens=False)

print(f"elapsed_ms: {elapsed_ms}")
print("\n=== OUTPUT ===\n")
print(text)


## Notes

- If inference is too slow, reduce `MAX_NEW_TOKENS` and `MAX_IMAGE_SIZE`.
- If you hit Windows local memory/pagefile issues, run this notebook on Kaggle/Colab GPU.
- For production-like serving, use the server mode in `run_multimodal_inference.py` and call `/generate`.
